# Diagnóstico del modelo de ML — Previsión de demanda (`demand_forecast`)

Notebook de diagnóstico para el caso de uso **demand** (etiqueta `n_pickups`).

Sin reentrenamiento: carga el modelo `@champion` del Model Registry de MLflow y lo analiza sobre las tablas gold.

> Uno de los tres notebooks de diagnóstico por caso de uso (`fare` / `duration` / `demand`). `USE_CASE` se fija más abajo; el cuerpo del análisis es idéntico en los tres.

Secciones:
1. Parámetros + configuración (Spark + MLflow).
2. Carga del modelo `@champion` + metadatos registrados.
3. Reconstrucción de los splits (`train` / `val` / `common_test`) y medición de cada uno.
4. Desglose por año — TRAIN_VAL_YEARS (casi in-sample) vs TEST_YEAR (reservado, la señal de drift).
5. Predicho vs Real en `common_test`.
6. Residuos.
7. Importancia de las features (independiente del algoritmo).
8. Fase 2 — Más datos A vs B.
9. Baseline vs champion.
10. Conclusiones.

## 1. Parámetros

`USE_CASE` está fijado a `demand` en este notebook — uno de los tres diagnósticos por caso de uso.

In [ ]:
USE_CASE = "demand"   # fijado para este notebook

from ml.train import USE_CASES
from ml.splits import TEST_YEAR, TRAIN_VAL_YEARS

cfg = USE_CASES[USE_CASE]
EXPERIMENT = cfg["experiment"]
SOURCE     = cfg["source"]
LABEL      = cfg["label_col"]
YEAR_COL   = cfg["year_col"]
print(f"USE_CASE={USE_CASE}  EXPERIMENT={EXPERIMENT}  LABEL={LABEL}")
print(f"Split: TRAIN_VAL_YEARS={TRAIN_VAL_YEARS}  TEST_YEAR={TEST_YEAR}")

In [ ]:
import mlflow
import mlflow.spark
from mlflow import MlflowClient
from pyspark.ml.feature import VectorAssembler
from pyspark.sql.functions import col
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from ml.splits import add_cyclical_features, make_splits
from ml.tracking import (
    compute_baseline_metrics,
    evaluate_predictions,
    evaluate_regression,
    get_ml_spark,
    setup_mlflow,
)

spark = get_ml_spark(f"ml-{USE_CASE}-diagnostics")
setup_mlflow(EXPERIMENT)
client = MlflowClient()
spark.version

## 2. Carga del modelo `@champion` desde el Registry

Resolver a través del alias `@champion` devuelve siempre la última versión promovida. Para versiones anteriores, sustituye `"champion"` por el número de versión:
`client.get_model_version(EXPERIMENT, "3")`.

In [ ]:
mv = client.get_model_version_by_alias(name=EXPERIMENT, alias="champion")
source_run = client.get_run(mv.run_id)
print(f"Registered model: {EXPERIMENT} v{mv.version}")
print(f"Source run ID:    {mv.run_id}")
print(f"Run name:         {source_run.data.tags.get('mlflow.runName')}")
print(f"Algorithm:        {source_run.data.tags.get('algorithm')}")
print(f"Phase:            {source_run.data.tags.get('phase')}")
print(f"Training data:    {source_run.data.tags.get('training_data')}")
print(f"Created by:       {source_run.data.tags.get('created_by')}")
print(f"Started:          {pd.Timestamp(source_run.info.start_time, unit='ms')}")
print(f"\nDescription:\n  {mv.description or '(none)'}\n")
print("Version tags:")
for k, v in sorted(mv.tags.items()):
    print(f"  {k}: {v}")

In [ ]:
metrics_df = pd.DataFrame.from_dict(source_run.data.metrics, orient='index', columns=['value'])
params_df  = pd.DataFrame.from_dict(source_run.data.params,  orient='index', columns=['value'])
display(metrics_df.sort_index())
display(params_df.sort_index())

In [ ]:
model_uri = f"models:/{EXPERIMENT}@champion"
# mlflow.spark.load_model deja el modelo en una ruta /tmp local del driver que los
# executors de los workers no ven (el FS por defecto del clúster es file://, así que MLflow nunca
# copia a dfs_tmpdir) -> PipelineModel.load lanza FileNotFoundException en los
# executors. En su lugar descargamos el modelo y dejamos su carpeta sparkml en el bind
# mount compartido ./src (misma ruta en cada contenedor, workers en solo lectura) y cargamos vía file://.
import os, shutil
from mlflow.artifacts import download_artifacts
from pyspark.ml import PipelineModel

local_model = download_artifacts(artifact_uri=model_uri)
shared_dir = "/opt/spark/src/.tmp_ingest/champion_sparkml"
shutil.rmtree(shared_dir, ignore_errors=True)
shutil.copytree(os.path.join(local_model, "sparkml"), shared_dir)
model = PipelineModel.load(f"file://{shared_dir}")
for i, stage in enumerate(model.stages):
    print(f"{i}: {type(stage).__name__}")

## 3. Reconstrucción de los splits + métricas por split

`make_splits` usa `seed=42` (el mismo que el script de entrenamiento), así que train / val / common_test son reproducibles. También reconstruimos las mismas columnas derivadas (sin/cos cíclicos para los algoritmos que no son árboles, filtro de etiqueta para `duration`) **y reaplicamos el mismo muestreo de Bernoulli** (`sample_fraction` de los params del run de origen, `seed=42`) para que los recuentos de filas aquí coincidan con los tags de la versión del momento de entrenamiento.

In [ ]:
# Reaplicamos EL MISMO muestreo (fracción + seed=42) con el que se entrenó el champion,
# para que los recuentos reconstruidos de train/val/common_test coincidan con los tags
# de la versión registrada. La fracción está en los params del run de origen como
# `sample_fraction`; los runs antiguos (previos al muestreo) no la tienen -> se leen como datos completos.
cols_to_keep = [LABEL, *cfg["numerical_cols"], *cfg["categorical_cols"]]
df = spark.table(SOURCE).select(*cols_to_keep)
if cfg["label_filter_predicate"] is not None:
    df = df.filter(cfg["label_filter_predicate"](col(LABEL)))

run_sample_fraction = source_run.data.params.get("sample_fraction")
if run_sample_fraction is not None:
    sf = float(run_sample_fraction)
    if 0.0 < sf < 1.0:
        df = df.sample(sf, seed=42)
        print(f"Reapplied training-time sample: fraction={sf:.2f}, seed=42")
    else:
        print(f"sample_fraction={sf} -> no sampling.")
else:
    print("Source run has no sample_fraction param (pre-sampling vintage) -> using full data.")

df, _ = add_cyclical_features(df, cfg["numerical_cols"])
train, val, common_test = make_splits(df, YEAR_COL)
for s in (train, val, common_test):
    s.cache()

print("Split sizes:")
for name, s in (('train', train), ('val', val), ('common_test', common_test)):
    print(f"  {name:<12} {s.count():>12,} rows")

print(f"\ntrain + val = TRAIN_VAL_YEARS {TRAIN_VAL_YEARS}")
print(f"common_test = TEST_YEAR {TEST_YEAR} (fully held out)")

print("\nChampion metrics per split:")
print(f"  {'split':<12} {'rmse':>12} {'mae':>12} {'r2':>12}")
for name, s in (('train', train), ('val', val), ('common_test', common_test)):
    m = evaluate_regression(model, s, LABEL)
    print(f"  {name:<12} {m['rmse']:>12.4f} {m['mae']:>12.4f} {m['r2']:>12.4f}")

## 4. Desglose por año — TRAIN_VAL_YEARS (casi in-sample) vs TEST_YEAR (señal de drift)

Con el split de holdout del último año, la señal honesta fuera de tiempo es la métrica sobre `common_test = TEST_YEAR` ({TEST_YEAR}). Para ponerla en contexto, puntuamos el champion en cada año de la tabla gold y comparamos:

- Las filas de **TRAIN_VAL_YEARS** se han visto en su mayoría durante el entrenamiento, así que su RMSE se acerca a un valor *in-sample* (una pequeña porción de cada año está en `val`, pero la mayoría está en `train`). Muestran cómo de bien ajusta el modelo la distribución de entrenamiento.
- Las filas de **TEST_YEAR** quedan totalmente reservadas — nunca se ven durante el entrenamiento ni la selección. La diferencia entre el RMSE de TEST_YEAR y los de TRAIN_VAL_YEARS es la señal de drift: una diferencia pequeña significa que el modelo generaliza al año futuro; una grande, que el año de test se aleja de la distribución de entrenamiento.

In [ ]:
# Puntuamos el champion en cada año — union(train, val, common_test) cubre
# toda la tabla gold con las columnas proyectadas con las que entrenamos.
full_df = train.unionByName(val).unionByName(common_test)
year_pred = model.transform(full_df).select(YEAR_COL, LABEL, 'prediction').cache()
years = sorted(r[YEAR_COL] for r in year_pred.select(YEAR_COL).distinct().collect())
rows = []
for y in years:
    yr = year_pred.filter(col(YEAR_COL) == y).select(LABEL, 'prediction')
    n = yr.count()
    if n == 0:
        continue
    m = evaluate_predictions(yr, LABEL)
    rows.append({
        'year': y,
        'role': 'TEST (held out)' if y == TEST_YEAR else 'TRAIN_VAL (in-sample-ish)',
        'n': n,
        **m,
    })
year_pred.unpersist()
per_year = pd.DataFrame(rows)
display(per_year)

# Diagrama de barras. TEST_YEAR resaltado en rojo para que el salto de drift se vea a simple vista.
bar_colors = ['#d62728' if y == TEST_YEAR else '#1f77b4' for y in per_year['year']]
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, metric in zip(axes, ('rmse', 'mae', 'r2')):
    ax.bar(per_year['year'].astype(str), per_year[metric], color=bar_colors)
    ax.set_xlabel('year')
    ax.set_ylabel(metric)
    ax.set_title(f'{metric} per year (red = TEST_YEAR)')
plt.tight_layout()

## 5. Predicho vs Real en `common_test`

Una muestra del 5 % de `common_test` traída a pandas para graficar. Los puntos deberían alinearse cerca de `y = x`; una curvatura o un sesgo sistemático indica mala calibración.

In [ ]:
sample_fraction = 0.05
pred_full = model.transform(common_test).select(LABEL, 'prediction')
pdf = pred_full.sample(sample_fraction, seed=42).toPandas()
pdf['residual'] = pdf[LABEL] - pdf['prediction']
print(f"common_test sample: {len(pdf):,} rows ({sample_fraction:.0%})")
pdf.describe()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(pdf[LABEL], pdf['prediction'], alpha=0.08, s=6)
lim = float(max(pdf[LABEL].max(), pdf['prediction'].max()))
ax.plot([0, lim], [0, lim], 'r--', linewidth=1, label='y = x')
ax.set_xlabel(f'actual {LABEL}')
ax.set_ylabel(f'predicted {LABEL}')
ax.set_title(f'Predicted vs Actual \u2014 {EXPERIMENT} (common_test sample)')
ax.set_xlim(0, lim)
ax.set_ylim(0, lim)
ax.legend()
plt.tight_layout()

## 6. Residuos

- **Residuo vs predicción** — bandas horizontales = error homocedástico (bien); un embudo = heterocedasticidad.
- **Histograma de residuos** — debería estar centrado en 0 y ser simétrico. La asimetría = sobre/infra-predicción sistemática.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].scatter(pdf['prediction'], pdf['residual'], alpha=0.08, s=6)
axes[0].axhline(0, color='r', linestyle='--', linewidth=1)
axes[0].set_xlabel('predicted')
axes[0].set_ylabel('residual (actual \u2212 predicted)')
axes[0].set_title('Residual vs prediction')
axes[1].hist(pdf['residual'], bins=80, edgecolor='black', linewidth=0.3)
axes[1].axvline(0, color='r', linestyle='--', linewidth=1)
axes[1].set_xlabel('residual')
axes[1].set_ylabel('frequency')
axes[1].set_title('Residual distribution')
plt.tight_layout()

## 7. Importancia de las features

- **GBT / RFR** — `featureImportances` (reducción media de impureza). Ambos ensembles de árboles exponen el mismo atributo.
- **GLR** — `abs(coefficients)`. Las features de GLR están estandarizadas (StandardScaler), así que las magnitudes son comparables entre columnas.

In [ ]:
final = model.stages[-1]
algorithm = type(final).__name__
asm = next((s for s in model.stages if isinstance(s, VectorAssembler) and s.getOutputCol() == 'assembled_features'), None)
inputs = list(asm.getInputCols()) if asm is not None else None

if hasattr(final, 'featureImportances'):
    values = final.featureImportances.toArray(); metric_name = 'importance'
elif hasattr(final, 'coefficients'):
    values = abs(final.coefficients.toArray()); metric_name = 'abs_coef'
else:
    raise RuntimeError(f'No featureImportances/coefficients available for {algorithm}')

label_col = 'feature' if inputs is not None and len(inputs) == len(values) else 'feature_idx'
labels = inputs if label_col == 'feature' else list(range(len(values)))
fi = (
    pd.DataFrame({label_col: labels, metric_name: values})
    .sort_values(metric_name, ascending=False)
    .reset_index(drop=True)
)
top = fi.head(30)
display(top)

fig, ax = plt.subplots(figsize=(8, max(3, 0.3 * len(top))))
sns.barplot(data=top, x=metric_name, y=label_col, ax=ax, color='steelblue')
ax.set_title(f'Top feature importances — {EXPERIMENT} ({algorithm})')
plt.tight_layout()

## 8. Fase 2 — Más datos: A vs B

Carga el run `more-data-A` (el ganador reentrenado sobre `SUBSET_A_YEARS`, el subconjunto inicial de años de entrenamiento) y el ganador de la Fase 1 (= B, entrenado sobre todo `TRAIN_VAL_YEARS`) a partir de su tag `more_data_subset`, y compara su `common_test_rmse`. Ambos se puntúan sobre el mismo año reservado (`TEST_YEAR`), así que ninguno ve el año de test — el contraste aísla "¿ayudan más datos de entrenamiento a generalizar a un año futuro?". Si B supera a A de forma apreciable, los años extra de entrenamiento merecen la pena; si quedan igualados, el modelo ya está saturado con los primeros años.

Nota: train.py etiqueta al ganador de la Fase 1 con `more_data_subset='B'` en vez de reentrenarlo — el reentrenamiento sería idéntico, así que reutilizamos el run existente.

In [ ]:
exp = client.get_experiment_by_name(EXPERIMENT)
# Los filtros de búsqueda de MLflow no admiten ni OR ni IN sobre tags, así que consultamos cada
# subconjunto por separado y concatenamos.
def _latest_more_data(subset):
    return client.search_runs(
        [exp.experiment_id],
        filter_string=f"tags.more_data_subset = '{subset}' and attributes.status = 'FINISHED'",
        order_by=['attributes.start_time DESC'], max_results=10,
    )
md_runs = _latest_more_data('A') + _latest_more_data('B')
md = pd.DataFrame([
    {
        'run_name':         r.data.tags.get('mlflow.runName'),
        'subset':           r.data.tags.get('more_data_subset'),
        'training_data':    r.data.tags.get('training_data'),
        'algorithm':        r.data.tags.get('algorithm'),
        'val_rmse':         r.data.metrics.get('val_rmse'),
        'common_test_rmse': r.data.metrics.get('common_test_rmse'),
        'common_test_mae':  r.data.metrics.get('common_test_mae'),
        'common_test_r2':   r.data.metrics.get('common_test_r2'),
        'start_time':       pd.Timestamp(r.info.start_time, unit='ms'),
    } for r in md_runs
])
# Nos quedamos solo con el run más reciente por subconjunto para comparar en igualdad de condiciones.
latest_pair = md.sort_values('start_time', ascending=False).drop_duplicates(subset='subset').head(2)
display(latest_pair[['subset', 'training_data', 'val_rmse',
                     'common_test_rmse', 'common_test_mae', 'common_test_r2']])

if len(latest_pair) == 2:
    a = latest_pair[latest_pair['subset'] == 'A'].iloc[0]
    b = latest_pair[latest_pair['subset'] == 'B'].iloc[0]
    delta = b['common_test_rmse'] - a['common_test_rmse']
    pct = 100 * delta / a['common_test_rmse']
    verdict = ('B better' if delta < 0 else 'A better' if delta > 0 else 'tie')
    print(f"\nA → B common_test_rmse change: {delta:+.4f} ({pct:+.2f}%) — {verdict}")

## 9. Baseline vs champion

El baseline predice `mean(label)` por `(group_cols)` (p. ej. `(pu_location_id, pickup_hour)`) calculado sobre el conjunto de entrenamiento, recurriendo a la media global para los grupos no vistos en `common_test`. **Media — no mediana — porque reportamos RMSE**: la media por grupo es el predictor constante por grupo óptimo en RMSE (la mediana lo es para MAE). Cualquier valor por encima de este suelo en `common_test_rmse` no está aprendiendo más allá de las medias por grupo; el champion debería superarlo por un margen apreciable.

In [ ]:
baseline = compute_baseline_metrics(
    train_df=train, test_df=common_test,
    label_col=LABEL, group_cols=cfg['baseline_group_cols'],
)
champ = evaluate_regression(model, common_test, LABEL)
cmp = pd.DataFrame([
    {'model': 'baseline (mean-per-group)', **baseline},
    {'model': f'champion v{mv.version}',   **champ},
])
display(cmp[['model', 'rmse', 'mae', 'r2']])

delta = champ['rmse'] - baseline['rmse']
pct = 100 * delta / baseline['rmse']
print(f"\nchampion − baseline common_test_rmse: {delta:+.4f} ({pct:+.2f}%)")

## 10. Conclusiones — Previsión de demanda

> _Completa la lectura cualitativa del modelo:_
>
> - ¿`val` ↔ `common_test` muestra sobreajuste por selección? (§3 — comparar estimaciones puntuales)
> - Desglose por año: ¿es `TEST_YEAR` drásticamente peor que los `TRAIN_VAL_YEARS`? (§4)
> - ¿El modelo está bien calibrado en `common_test`? (§5)
> - ¿Heterocedasticidad / sesgo en los residuos? (§6)
> - ¿Qué features dominan? ¿Coincide con lo esperado en el dominio? (§7)
> - ¿Añadir más años de entrenamiento (B vs A) ayuda en el año reservado? (§8)
> - ¿El champion supera al baseline por un margen apreciable? (§9)
> - Próximos pasos / mejoras.